#Extracting Signs, Categorizing and Classifying them
## Newest 2026 Edition of CA MUTCD

In [6]:
import os
import re
import requests
import pandas as pd
import pypdf
from openai import OpenAI

# ============================================================
# 0. OPENAI CLIENT (KEY FROM WINDOWS ENV)
# ============================================================

def get_openai_client():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError("OPENAI_API_KEY not found in environment variables.")
    return OpenAI(api_key=api_key)


# ============================================================
# 1. DOWNLOAD PDFs (Chapters 2B–2N)
# ============================================================

BASE_URL = "https://dot.ca.gov/-/media/dot-media/programs/safety-programs/documents/camutcd/2026/"
CHAPTERS = [f"2{c}" for c in "BCDEFGHIJKLMN"]

def download_pdfs(output_dir="pdfs"):
    os.makedirs(output_dir, exist_ok=True)

    for chapter in CHAPTERS:
        pdf_name = f"chapter-{chapter}.pdf"
        url = f"{BASE_URL}{pdf_name}"

        print(f"Downloading {pdf_name}...")
        r = requests.get(url)

        if r.status_code == 200:
            with open(os.path.join(output_dir, pdf_name), "wb") as f:
                f.write(r.content)
        else:
            print(f"Failed: {pdf_name} ({r.status_code})")


# ============================================================
# 2. EXTRACT & CONCATENATE TEXT
# ============================================================

def extract_all_text(pdf_dir="pdfs"):
    master_text = ""

    for file in sorted(os.listdir(pdf_dir)):
        if file.endswith(".pdf"):
            path = os.path.join(pdf_dir, file)
            print(f"Extracting {file}...")

            with open(path, "rb") as f:
                reader = pypdf.PdfReader(f)
                for page in reader.pages:
                    master_text += page.extract_text() + "\n"

    with open("master_text.txt", "w", encoding="utf-8") as f:
        f.write(master_text)

    return master_text


# ============================================================
# 3. EXTRACT SIGN CODES
# ============================================================

SIGN_REGEX = r"""
(
 R\d+[A-Z]?-\d+[A-Z]? |
 W\d+[A-Z]?-\d+[A-Z]? |
 M\d+[A-Z]?-\d+[A-Z]? |
 D\d+[A-Z]?-\d+[A-Z]? |
 E\d+[A-Z]?-\d+[A-Z]? |
 I\d+[A-Z]?-\d+[A-Z]? |
 G\d+[A-Z]?-\d+[A-Z]? |
 J\d+[A-Z]?-\d+[A-Z]? |
 K\d+[A-Z]?-\d+[A-Z]? |
 L\d+[A-Z]?-\d+[A-Z]? |
 N\d+[A-Z]?-\d+[A-Z]? |
 SR\d{3}\(CA\) |
 S\d{3}\(CA\)
)
"""

def extract_sign_codes(text):
    codes = re.findall(SIGN_REGEX, text, re.VERBOSE)
    codes = sorted(set(codes))

    with open("sign_codes.txt", "w") as f:
        f.write("\n".join(codes))

    return codes


# ============================================================
# 4. AUTO-CLASSIFY (Shall / Should / May)
# ============================================================

WINDOW = 200

def classify(text, code):
    idx = text.find(code)
    if idx == -1:
        return "Unclassified", ""

    snippet = text[max(0, idx-WINDOW): idx+WINDOW]

    if "shall" in snippet.lower():
        return "Mandatory", snippet
    if "should" in snippet.lower():
        return "Recommended", snippet
    if "may" in snippet.lower():
        return "Optional", snippet

    return "Unclassified", snippet


def classify_all(text, codes):
    rows = []
    for code in codes:
        status, snippet = classify(text, code)
        rows.append([code, status, snippet])

    df = pd.DataFrame(rows, columns=["Sign Code", "Classification", "Context"])
    df.to_csv("classification.csv", index=False)
    return df


# ============================================================
# 5. ASSIGN REFERENCE SECTIONS
# ============================================================

SECTION_REGEX = r"(Section 2[A-N]\.\d+)"

def map_sections(text):
    sections = []
    for match in re.finditer(SECTION_REGEX, text):
        sections.append((match.start(), match.group()))
    return sections

def find_section(sections, index):
    prev = None
    for pos, sec in sections:
        if pos > index:
            break
        prev = sec
    return prev

def assign_sections(text, df):
    sections = map_sections(text)

    new_sections = []
    for code in df["Sign Code"]:
        idx = text.find(code)
        sec = find_section(sections, idx)
        new_sections.append(sec)

    df["Reference Section"] = new_sections
    df["Chapter"] = df["Reference Section"].str.extract(r"(2[A-N])")
    df.to_csv("sections.csv", index=False)
    return df


# ============================================================
# 6. POPULATE SIZE REFERENCES
# ============================================================

SIZE_TABLES = {
    "2B": "Table 2B-1",
    "2C": "Table 2C-2",
    "2D": "Table 2D-1",
}

def assign_size(chapter):
    return SIZE_TABLES.get(chapter, "SHS")

def add_size_tables(df):
    df["Size Table"] = df["Chapter"].apply(assign_size)
    df.to_csv("sizes.csv", index=False)
    return df


# ============================================================
# 7. EXTRACT SIGN NAMES
# ============================================================

NAME_REGEX = r"([A-Za-z ]+)\s*\(\s*{}"

def extract_name(text, code):
    pattern = NAME_REGEX.format(re.escape(code))
    m = re.search(pattern, text)
    return m.group(1).strip() if m else ""

def add_sign_names(text, df):
    df["Sign Name"] = df["Sign Code"].apply(lambda c: extract_name(text, c))
    df.to_csv("names.csv", index=False)
    return df


# ============================================================
# 8. ASSIGN MUTCD CATEGORY
# ============================================================

CATEGORY_MAP = {
    "2B": "Regulatory",
    "2C": "Warning",
    "2D": "Guide",
    "2E": "Freeway Guide",
    "2F": "Toll Road",
    "2G": "Managed Lane",
    "2H": "General Information",
    "2I": "General Service",
    "2J": "Specific Service",
    "2K": "Tourist Directional",
    "2L": "CMS",
    "2M": "Recreation & Cultural",
    "2N": "Emergency Management",
}

def add_categories(df):
    df["Category"] = df["Chapter"].map(CATEGORY_MAP)
    df.to_csv("categorized.csv", index=False)
    return df


# ============================================================
# 9. BUILD FINAL MULTI-SHEET EXCEL
# ============================================================

def build_excel(df):
    with pd.ExcelWriter("2026_CAMUTCD_2B_to_2N_Signs.xlsx") as writer:

        df.to_excel(writer, sheet_name="ALL_SIGNS", index=False)

        for chapter in sorted(df["Chapter"].dropna().unique()):
            df[df["Chapter"] == chapter].to_excel(writer, sheet_name=chapter, index=False)

        for cat in sorted(df["Category"].dropna().unique()):
            df[df["Category"] == cat].to_excel(writer, sheet_name=cat[:31], index=False)

        df["Sign Code"].to_excel(writer, sheet_name="CODES_ONLY", index=False)

        df[df["Classification"] == "Unclassified"].to_excel(writer, sheet_name="UNCLASSIFIED", index=False)


# ============================================================
# 10. MASTER RUNNER
# ============================================================

def run_all():
    print("\n=== STEP 1: Download PDFs ===")
    download_pdfs()

    print("\n=== STEP 2: Extract Text ===")
    text = extract_all_text()

    print("\n=== STEP 3: Extract Sign Codes ===")
    codes = extract_sign_codes(text)

    print("\n=== STEP 4: Classify (Shall/Should/May) ===")
    df = classify_all(text, codes)

    print("\n=== STEP 5: Assign Sections ===")
    df = assign_sections(text, df)

    print("\n=== STEP 6: Add Size Tables ===")
    df = add_size_tables(df)

    print("\n=== STEP 7: Extract Sign Names ===")
    df = add_sign_names(text, df)

    print("\n=== STEP 8: Assign Categories ===")
    df = add_categories(df)

    print("\n=== STEP 9: Build Excel ===")
    build_excel(df)

    print("\n=== COMPLETE: 2026_CAMUTCD_2B_to_2N_Signs.xlsx generated ===")


if __name__ == "__main__":
    run_all()


=== STEP 1: Download PDFs ===


invalid pdf header: b'\r\n\r\n\r'
EOF marker not found



=== STEP 2: Extract Text ===
Extracting chapter-2B.pdf...


PdfStreamError: Stream has ended unexpectedly

In [9]:
import os

for file in os.listdir("CA_MUTCD_2026_PDFs"):
    path = os.path.join("CA_MUTCD_2026_PDFs", file)
    size = os.path.getsize(path)
    if size < 50_000:  # <50 KB is almost always broken
        print("⚠️ Suspicious file:", file, "size:", size)


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'CA_MUTCD_2026_PDFs'